# F3 — Rendimiento de la lectura de datos
**Aporte de Mauricio Cid · Grupo 5**

Este notebook compara tres formas de cargar el CSV de licitaciones con `LectorCSV`. Mide tiempo y memoria, comprueba que los datos necesarios no cambian y justifica qué opción conviene. Está pensado para integrarse al notebook principal de F3.

**Recorrido:** entorno → opciones y columnas → herramientas → equivalencia → evidencia → tiempos → crecimiento → memoria → flujo completo → decisión.

## 1. Entorno y presentación

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
raiz = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/datos.py').exists())
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))
from src.datos import COLUMNAS_ANALISIS, TIPOS_ANALISIS, sha256_archivo
from src.entorno import versiones_entorno
from F3.medir_lectura import RUTA, SHA256_CSV, crear_lectores, comprobar_equivalencia, sha256_codigo

Funciones de presentación: solo cambian etiquetas y formato visible, igual que en `F3_Algoritmos.ipynb`.

In [2]:
NOMBRES_OPCION = {'actual': 'Actual (todas las columnas)', 'solo_columnas': 'Solo columnas necesarias',
                  'columnas_y_tipos': 'Columnas y tipos definidos'}
MIB = 1024 ** 2

def mostrar_tabla(tabla, titulo, formatos=None):
    estilo = (tabla.style.hide(axis='index')
              .format(formatos or {}, na_rep='—', decimal=',', thousands='.', escape='html')
              .set_caption(titulo)
              .set_table_styles([
                  {'selector': 'caption', 'props': [('text-align', 'left'), ('font-weight', 'bold'), ('padding', '12px 0 6px')]},
                  {'selector': 'th', 'props': [('text-align', 'left'), ('padding', '8px 10px'), ('white-space', 'normal'), ('border-bottom', '2px solid #94a3b8')]},
                  {'selector': 'td', 'props': [('padding', '7px 10px'), ('text-align', 'right'), ('vertical-align', 'top'), ('font-variant-numeric', 'tabular-nums'), ('border-bottom', '1px solid #cbd5e1')]},
                  {'selector': 'td.col0', 'props': [('text-align', 'left'), ('max-width', '320px'), ('white-space', 'normal')]},
              ]))
    display(estilo)

print(sys.version)
mostrar_tabla(pd.DataFrame(list(versiones_entorno().items()), columns=['Dependencia', 'Versión']), 'Versiones del entorno')

3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


Dependencia,Versión
numpy,2.3.5
pandas,3.0.1
jupyterlab,4.6.4
ipykernel,7.3.0
nbformat,5.11.1
nbconvert,7.17.1
nbclient,0.11.0


## 2. Opciones comparadas y columnas acordadas
El CSV tiene 44.226 filas y 74 columnas (~72 MB). F1 y F2 exploran todas las columnas; el análisis de F3 solo necesita algunas.

| Opción | Configuración de `LectorCSV` |
| --- | --- |
| Actual | Sin cambios: lee las 74 columnas y pandas infiere los tipos. |
| Solo columnas necesarias | `columnas=COLUMNAS_ANALISIS` (`usecols` de pandas). |
| Columnas y tipos definidos | Además `tipos=TIPOS_ANALISIS` (`dtype`): `category` para las cuatro categóricas. |

Las columnas se acordaron según lo que exige el código existente, no solo el cálculo:

In [3]:
motivo = {
    'NroLicitacion': 'Columna clave exigida por LimpiadorLicitaciones',
    'TipoLicitacion': 'Agrupación de proporciones',
    'TamanoProveedor': 'Agrupación de proporciones',
    'ResultadoOferta': 'Numerador y denominador de las proporciones',
    'EstadoLicitacion': 'Filtro de licitaciones adjudicadas',
    'FechaPublicacion': 'ReglaFechas y plazo_cierre_dias',
    'FechaCierre': 'ReglaFechas y plazo_cierre_dias',
}
mostrar_tabla(pd.DataFrame({'Columna': COLUMNAS_ANALISIS,
                            'Tipo al leer': [TIPOS_ANALISIS.get(c, 'inferido') for c in COLUMNAS_ANALISIS],
                            'Motivo': [motivo[c] for c in COLUMNAS_ANALISIS]}), 'Columnas leídas por las opciones nuevas')

Columna,Tipo al leer,Motivo
NroLicitacion,inferido,Columna clave exigida por LimpiadorLicitaciones
TipoLicitacion,category,Agrupación de proporciones
TamanoProveedor,category,Agrupación de proporciones
ResultadoOferta,category,Numerador y denominador de las proporciones
EstadoLicitacion,category,Filtro de licitaciones adjudicadas
FechaPublicacion,inferido,ReglaFechas y plazo_cierre_dias
FechaCierre,inferido,ReglaFechas y plazo_cierre_dias


Las fechas no se tipan al leer: la limpieza ya las convierte con formato mixto y excluye el año 1900; repetir esa lógica en la lectura la duplicaría. `NroLicitacion` conserva el tipo inferido porque la limpieza no lo normaliza y así los datos limpios quedan idénticos a la lectura actual.

## 3. Herramientas de medición

| Herramienta | Qué mide | Uso aquí |
| --- | --- | --- |
| `timeit` (basado en `perf_counter`) | Tiempo transcurrido, con GC desactivado | 7 rondas por opción, orden alternado con semilla 2026 |
| `DataFrame.memory_usage(deep=True)` | Tamaño del DataFrame resultante, incluyendo textos | Métrica principal de memoria |
| `tracemalloc` | Pico de asignaciones rastreadas por Python | 3 ejecuciones separadas del tiempo |

Se reutilizan `medir_tiempos`, `medir_memoria` y `resumir_medicion` de `medir_algoritmos.py`, para que ambos experimentos sigan la misma metodología. `tracemalloc` no equivale a la memoria total del proceso ni ve todos los búferes internos del lector de pandas (Python Software Foundation, s. f.-b). Por eso el tamaño del DataFrame es la referencia principal.

## 4. Equivalencia antes de medir
Una lectura más rápida que entrega otros datos no es una optimización. Cada opción se lee, se limpia y se valida. Luego se comparan con la lectura actual las siete columnas acordadas, las tres variables derivadas y las tablas de proporciones por tamaño y tipo. La comparación se hace después de limpiar: la limpieza convierte los textos a `string`, y la opción con tipos entrega `category` al leer.

In [4]:
equivalencia = comprobar_equivalencia(crear_lectores(), RUTA)
vista = pd.DataFrame(equivalencia)
vista['opcion'] = vista.opcion.map(NOMBRES_OPCION)
mostrar_tabla(vista.rename(columns={'opcion': 'Opción', 'filas': 'Filas', 'columnas_limpias': 'Columnas tras limpiar',
                                    'columnas_necesarias': 'Columnas necesarias', 'validacion': 'Validación',
                                    'proporciones': 'Proporciones'}),
              'Equivalencia con la lectura actual', {'Filas': '{:,.0f}'})

Opción,Filas,Columnas tras limpiar,Columnas necesarias,Validación,Proporciones
Actual (todas las columnas),44.226,74,iguales,OK,iguales
Solo columnas necesarias,44.226,10,iguales,OK,iguales
Columnas y tipos definidos,44.226,10,iguales,OK,iguales


## 5. Evidencia utilizada
Por defecto se muestran las mediciones guardadas. Antes se comprueban los hashes del CSV, de `src/datos.py` y de los scripts de medición. En los archivos de código, el hash se calcula con saltos de línea normalizados: git en Windows los convierte a CRLF, y sin normalizar la misma versión tendría otro hash en cada equipo. Para medir de nuevo, activar `REGENERAR_MEDICIONES`; si cambia alguno de esos archivos, es obligatorio hacerlo. Una nueva ejecución produce tiempos algo distintos.

In [5]:
from F3.medir_lectura import ejecutar
REGENERAR_MEDICIONES = False  # True repite el experimento y guarda nueva evidencia.
evidencia = raiz / 'F3/mediciones_lectura/mediciones.json'
if REGENERAR_MEDICIONES or not evidencia.exists():
    ejecutar()
registro = json.loads(evidencia.read_text(encoding='utf-8'))
assert registro['raw_sha256'] == sha256_archivo(RUTA) == SHA256_CSV
# Código: huella con saltos de línea normalizados, igual en Windows y en otros sistemas.
assert registro['datos_sha256'] == sha256_codigo(raiz / 'src/datos.py')
assert registro['medicion_sha256'] == sha256_codigo(raiz / 'F3/medir_lectura.py')
assert registro['medir_algoritmos_sha256'] == sha256_codigo(raiz / 'F3/medir_algoritmos.py')
print('Mediciones de:', registro['fecha_utc'])
print('Python:', registro['python'])
print('Plataforma:', registro['sistema'])
print('Repeticiones:', registro['repeticiones'], '| Ejecuciones por ronda:', registro['ejecuciones_por_repeticion'])
mediciones = pd.DataFrame(registro['mediciones'])
mediciones['Opción'] = mediciones.opcion.map(NOMBRES_OPCION)
lectura = mediciones.loc[mediciones.etapa.eq('lectura')]
completo = lectura.loc[lectura.filas.eq(lectura.filas.max())].set_index('opcion')
flujo = mediciones.loc[mediciones.etapa.eq('flujo_completo')].set_index('opcion')
print('Casos experimentales:', len(mediciones))

Mediciones de: 2026-09-27T18:07:37.141279+00:00
Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Plataforma: Windows-11-10.0.26200-SP0
Repeticiones: 7 | Ejecuciones por ronda: 1
Casos experimentales: 12


## 6. Tiempo de lectura del archivo completo
Se informan el mínimo (la ejecución con menos interferencia), la mediana y los cuartiles de siete rondas. Los cuartiles describen esta sesión; no son intervalos de confianza (Python Software Foundation, s. f.-a).

In [6]:
vista = pd.DataFrame({'Opción': completo['Opción'], 'Columnas': completo.columnas,
                      'Mínimo (ms)': completo.min_ms, 'Mediana (ms)': completo.mediana_ms,
                      'Q25 (ms)': completo.q25_ms, 'Q75 (ms)': completo.q75_ms,
                      'Reducción de mediana (%)': (1 - completo.mediana_ms / completo.loc['actual', 'mediana_ms']) * 100})
mostrar_tabla(vista, 'Lectura de 44.226 filas — tiempos',
              {'Columnas': '{:.0f}', **{c: '{:.1f}' for c in vista.columns if c not in ('Opción', 'Columnas')}})

Opción,Columnas,Mínimo (ms),Mediana (ms),Q25 (ms),Q75 (ms),Reducción de mediana (%)
Actual (todas las columnas),74,"828,4","844,5","831,6","982,3","0,0"
Solo columnas necesarias,7,"386,5","401,7","396,0","440,7","52,4"
Columnas y tipos definidos,7,"391,4","406,7","399,1","465,2","51,8"


Leer 7 de 74 columnas reduce el tiempo aproximadamente a la mitad, no a una décima parte. `usecols` evita crear las columnas descartadas, pero el lector igual debe recorrer y separar cada línea completa del archivo. Definir `category` no acelera la lectura: la diferencia con "solo columnas" está dentro de la dispersión entre rondas.

## 7. Cómo crece el tiempo con las filas
Una sola medición no muestra la complejidad; hay que ver cómo cambia el tiempo al aumentar el volumen.

In [7]:
vista = lectura.pivot(index='filas', columns='opcion', values='mediana_ms')[list(NOMBRES_OPCION)]
vista['Razón actual / solo columnas'] = vista['actual'] / vista['solo_columnas']
vista = vista.rename(columns=NOMBRES_OPCION).rename_axis('Filas').reset_index()
vista.columns.name = None
mostrar_tabla(vista, 'Medianas de lectura por tamaño (ms)',
              {'Filas': '{:,.0f}', **{c: '{:.1f}' for c in NOMBRES_OPCION.values()}, 'Razón actual / solo columnas': '{:.2f}'})

Filas,Actual (todas las columnas),Solo columnas necesarias,Columnas y tipos definidos,Razón actual / solo columnas
1.000,"22,4","15,0","16,0","1,49"
10.000,"205,2","106,5","113,3","1,93"
44.226,"844,5","401,7","406,7","2,10"


Las tres opciones crecen de forma aproximadamente proporcional a las filas: leer es O(N·C), con N filas y C columnas recorridas por el lector. La ventaja de leer menos columnas aumenta levemente con el volumen: en muestras pequeñas pesa el costo fijo de abrir el archivo e interpretar el encabezado.

## 8. Memoria
El tamaño del DataFrame es la diferencia más grande entre las opciones.

In [8]:
vista = pd.DataFrame({'Opción': completo['Opción'],
                      'DataFrame (MiB)': completo.memoria_dataframe_bytes / MIB,
                      'Pico tracemalloc (MiB)': completo.pico_trazado_bytes_mediana / MIB,
                      'Reducción del DataFrame (%)': (1 - completo.memoria_dataframe_bytes / completo.loc['actual', 'memoria_dataframe_bytes']) * 100})
mostrar_tabla(vista, 'Lectura de 44.226 filas — memoria', {c: '{:.1f}' for c in vista.columns if c != 'Opción'})

Opción,DataFrame (MiB),Pico tracemalloc (MiB),Reducción del DataFrame (%)
Actual (todas las columnas),"195,7","43,5","0,0"
Solo columnas necesarias,"20,6","4,4","89,5"
Columnas y tipos definidos,"8,9","3,2","95,5"


Leer solo las columnas necesarias reduce el DataFrame en torno a un 89 %. Con tipos `category` baja aún más, porque cada categoría repetida se guarda una vez y las filas almacenan códigos enteros en lugar de textos. Aquí está el intercambio: los tipos no cambian el tiempo, pero sí la memoria.

## 9. Flujo completo: lectura, limpieza y proporciones
Como el costo de construir un índice, el ahorro al leer solo importa si se mantiene en el uso real. Se mide la lectura, `LimpiadorLicitaciones` y las dos tablas de proporciones.

In [9]:
vista = pd.DataFrame({'Opción': flujo['Opción'], 'Mínimo (ms)': flujo.min_ms, 'Mediana (ms)': flujo.mediana_ms,
                      'DataFrame limpio (MiB)': flujo.memoria_dataframe_bytes / MIB,
                      'Razón actual / opción': flujo.loc['actual', 'mediana_ms'] / flujo.mediana_ms})
mostrar_tabla(vista, 'Flujo completo — 44.226 filas',
              {'Mínimo (ms)': '{:.1f}', 'Mediana (ms)': '{:.1f}', 'DataFrame limpio (MiB)': '{:.1f}', 'Razón actual / opción': '{:.2f}'})

Opción,Mínimo (ms),Mediana (ms),DataFrame limpio (MiB),Razón actual / opción
Actual (todas las columnas),"1195,6","1252,8","171,5","1,00"
Solo columnas necesarias,"482,0","501,7","17,6","2,50"
Columnas y tipos definidos,"488,0","492,9","17,6","2,54"


El ahorro crece en el flujo completo: la limpieza copia y transforma todas las columnas que recibe, así que menos columnas también la abaratan. En cambio, la ventaja de memoria de `category` desaparece tras limpiar: `normalizar_categorias` convierte esas columnas a `string`, y ambas opciones nuevas terminan con el mismo DataFrame limpio.

## 10. Decisión
**Para el análisis de F3 conviene leer con `columnas=COLUMNAS_ANALISIS` y `tipos=TIPOS_ANALISIS`.**

- **Tiempo:** la lectura del archivo completo baja de ~845 ms a ~407 ms en mediana, y el flujo completo de ~1.253 ms a ~493 ms (unas 2,5 veces más rápido).
- **Memoria:** el DataFrame leído pasa de ~196 MiB a ~9 MiB, y el limpio de ~172 MiB a ~18 MiB.
- **Equivalencia:** las columnas necesarias, la validación y las proporciones quedan idénticas a la lectura actual.
- **Por qué tipos y no solo columnas:** definir tipos no cuesta tiempo y reduce a menos de la mitad la memoria de la lectura. El beneficio se pierde en la limpieza actual; conservar `category` durante la limpieza queda como mejora posible, no realizada.
- **F1 y F2** mantienen la lectura completa, que sigue siendo el comportamiento por defecto de `LectorCSV`, porque exploran todas las columnas.

La medición cambia una decisión: la lectura deja de ser igual para todas las fases y pasa a seleccionarse según lo que cada análisis necesita.

**Limitaciones:** las cifras corresponden a un equipo, una sesión y el sistema de archivos con caché después del calentamiento; una primera lectura en frío puede tardar más. `tracemalloc` no mide la memoria total del proceso.

## 11. Reproducibilidad
Desde la raíz del repositorio:

```powershell
.\.venv\Scripts\python.exe -m unittest F3.test_lectura F3.test_nucleo_poo F3.test_algoritmos -v
.\.venv\Scripts\python.exe F3\medir_lectura.py
.\.venv\Scripts\python.exe F3\verificar_lectura.py
```

El script guarda `F3/mediciones_lectura/resumen.csv` y `mediciones.json` (observaciones, parámetros, entorno y hashes). El verificador ejecuta este notebook en un kernel nuevo y registra el resultado en `F3/mediciones_lectura/ejecucion.json`. Si se regeneran las mediciones, se deben revisar las cifras de la sección 10.

## 12. Referencias técnicas
- Python Software Foundation. (s. f.-a). *timeit — Measure execution time of small code snippets*. https://docs.python.org/3/library/timeit.html
- Python Software Foundation. (s. f.-b). *tracemalloc — Trace memory allocations*. https://docs.python.org/3/library/tracemalloc.html
- The pandas development team. (s. f.). *pandas.read_csv*. https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
- The pandas development team. (s. f.). *Categorical data*. https://pandas.pydata.org/docs/user_guide/categorical.html